In [0]:
 %run ../../02_common_utils/operations

In [0]:
# COMMAND ----------
catalog           = "charles_schwab_retailbrokerage_dev_team_lemma"
silver_watches    = f"{catalog}.silver.watches"
gold_dim_customer = f"{catalog}.gold.dim_customer"
gold_dim_security = f"{catalog}.gold.dim_security"
gold_fact_watches = f"{catalog}.gold.fact_watches"

dbutils.widgets.text("batch_id", "1", "Batch ID")
current_batch = dbutils.widgets.get("batch_id")



try:
    run_info_row = spark.sql(f"SELECT _run_id, _batch FROM {silver_watches} LIMIT 1").first()
    carried_run_id = run_info_row[0] if run_info_row else "unknown"
    carried_batch = run_info_row[1] if run_info_row else "unknown"
except Exception:
    carried_run_id = "unknown"
    carried_batch = "unknown"

print(f"carried_run_id: {carried_run_id}")
print(f"carried_batch: {carried_batch}")

In [0]:
log_pipeline_message(spark, carried_run_id, 'INFO', 'gold_customer_watches', 'Starting processing for gold fact_watches')
start_pipeline_run(spark, carried_run_id, carried_batch)
log_domain_run_status(spark, carried_run_id, carried_batch, 'CUSTOMER', 'RUNNING')

In [0]:
# AQE Optimization for Window Functions
spark.conf.set("spark.sql.adaptive.enabled", "true")

# Safety net: Mock security table if Market domain hasn't run yet
try:
    spark.read.table(gold_dim_security).createOrReplaceTempView("v_dim_security")
except Exception:
    spark.sql("SELECT CAST(NULL AS BIGINT) AS sk_securityid, '' AS symbol, CAST('1900-01-01' AS DATE) as effectivedate, CAST('9999-12-31' AS DATE) as enddate WHERE 1=0").createOrReplaceTempView("v_dim_security")

In [0]:
# ─── BUILD AND OVERWRITE ─────────────────────────────────────────────────
spark.sql(f"""
    WITH RankedEvents AS (
        SELECT 
            CAST(W_C_ID AS BIGINT) as w_c_id, W_S_SYMB as w_s_symb, CAST(W_DTS AS TIMESTAMP) as event_ts, W_ACTION,
            LEAD(W_ACTION) OVER (PARTITION BY W_C_ID, W_S_SYMB ORDER BY W_DTS) as next_action,
            LEAD(CAST(W_DTS AS TIMESTAMP)) OVER (PARTITION BY W_C_ID, W_S_SYMB ORDER BY W_DTS) as next_event_ts
        FROM {silver_watches}
    ),
    ActiveWatches AS (
        -- Keep only ACTV events. If the NEXT event is CNCL, mark that as the removed date.
        SELECT w_c_id, w_s_symb, event_ts as placed_ts,
            CASE WHEN next_action = 'CNCL' THEN next_event_ts ELSE NULL END as removed_ts
        FROM RankedEvents WHERE W_ACTION = 'ACTV'
    )
    SELECT
        aw.w_c_id, aw.w_s_symb, dc.sk_customerid, ds.sk_securityid,
        CAST(date_format(aw.placed_ts, 'yyyyMMdd') AS BIGINT) as sk_dateid_dateplaced,
        CAST(date_format(aw.removed_ts, 'yyyyMMdd') AS BIGINT) as sk_dateid_dateremoved,
        '{current_batch}' AS _batch, CURRENT_TIMESTAMP() AS _load_ts, '{carried_run_id}' AS _run_id
    FROM ActiveWatches aw
    -- Temporal Join: Find the customer version that was active ON the day the watch was placed
    LEFT JOIN {gold_dim_customer} dc 
        ON aw.w_c_id = dc.customerid AND aw.placed_ts >= dc.effectivedate AND aw.placed_ts < dc.enddate
    -- Temporal Join: Find the security version that was active ON the day the watch was placed
    LEFT JOIN v_dim_security ds 
        ON aw.w_s_symb = ds.symbol AND aw.placed_ts >= ds.effectivedate AND aw.placed_ts < ds.enddate
""").write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(gold_fact_watches)

fact_watches_count = spark.sql(f"SELECT COUNT(*) FROM {gold_fact_watches}").first()[0]
print(f"gold.fact_watches rows: {fact_watches_count} (Expected: 2,412,745 after B3)")


In [0]:
# ─── LOGGING ─────────────────────────────────────────────────────────────
silver_watches_count = spark.sql(f"SELECT COUNT(*) FROM {silver_watches}").first()[0]
log_audit_event(spark, carried_run_id, current_batch, "gold", "fact_watches", "OVERWRITE", fact_watches_count)
log_pipeline_recon(spark, carried_run_id, current_batch, "CUSTOMER", "fact_watches", "silver", "gold", silver_watches_count, fact_watches_count)